# Phase 3 — Fundamental Topology
**Steps 3.1 – 3.4** | Degree distribution, small-world test, centrality, hierarchical structure.

**Deliverable:** Stats table + degree distribution plot + small-world comparison.

In [ ]:
import os, pickle
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib
import powerlaw
import warnings
warnings.filterwarnings('ignore')

matplotlib_style = {'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
                    'axes.edgecolor': '#30363d', 'text.color': 'white',
                    'axes.labelcolor': 'white', 'xtick.color': 'white',
                    'ytick.color': 'white', 'figure.dpi': 150,
                    'grid.color': '#30363d', 'grid.alpha': 0.5}
plt.rcParams.update(matplotlib_style)

ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
NETS   = os.path.join(ROOT, 'results', 'networks')
PLOTS  = os.path.join(ROOT, 'results', 'plots')
TABLES = os.path.join(ROOT, 'results', 'tables')

def load_graph(name):
    with open(os.path.join(NETS, f'{name}.pkl'), 'rb') as f:
        return pickle.load(f)

G_full = load_graph('full')
print(f'Full network loaded: {G_full.number_of_nodes()} nodes, {G_full.number_of_edges()} edges')

## Step 3.1 – Degree Distribution & Power-Law Fit

In [ ]:
degrees = [d for _, d in G_full.degree()]
degrees_arr = np.array(degrees)

# ── Power-law fit ────────────────────────────────────────────────────────────
fit = powerlaw.Fit(degrees_arr, discrete=True, verbose=False)
gamma = fit.power_law.alpha
xmin  = fit.power_law.xmin
print(f'Power-law exponent γ = {gamma:.3f}')
print(f'xmin = {xmin}')

# Likelihood-ratio test vs exponential
R, p = fit.distribution_compare('power_law', 'exponential', normalized_ratio=True)
print(f'LR test vs exponential: R = {R:.3f}, p = {p:.4f}')
if R > 0 and p < 0.05:
    print('  → Power law is a significantly better fit than exponential')
else:
    print('  → Cannot reject exponential; network may not be scale-free')

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Degree Distribution — Full UNGA Voting Network', color='white', fontsize=13)

# Linear scale
axes[0].hist(degrees_arr, bins=30, color='#58a6ff', alpha=0.8, edgecolor='#0d1117')
axes[0].set_xlabel('Degree k'); axes[0].set_ylabel('Count')
axes[0].set_title('Degree Histogram', color='white')
axes[0].grid(True, alpha=0.3)

# Log-log CCDF
fit.plot_ccdf(ax=axes[1], color='#58a6ff', label='Empirical CCDF')
fit.power_law.plot_ccdf(ax=axes[1], color='#f85149', ls='--',
                        label=f'Power law (γ={gamma:.2f})')
fit.exponential.plot_ccdf(ax=axes[1], color='#3fb950', ls=':',
                          label='Exponential fit')
axes[1].set_xlabel('Degree k'); axes[1].set_ylabel('P(X ≥ k)')
axes[1].set_title('Log-Log CCDF', color='white')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'p3_degree_distribution.png'),
            bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved degree distribution plot')

## Step 3.2 – Small-World Test (ER Comparison)

In [ ]:
from tqdm import tqdm

# Real network stats (use giant component)
gcc_nodes = max(nx.connected_components(G_full), key=len)
G_gcc = G_full.subgraph(gcc_nodes).copy()
N = G_gcc.number_of_nodes()
M = G_gcc.number_of_edges()
p_er = M / (N * (N-1) / 2)

print(f'Giant component: {N} nodes, {M} edges, p_ER = {p_er:.4f}')

C_real = nx.average_clustering(G_gcc)
try:
    d_real = nx.average_shortest_path_length(G_gcc)
except:
    # Sample-based approximation for large N
    sample_nodes = np.random.choice(list(G_gcc.nodes), size=min(50, N), replace=False)
    lengths = []
    for s in sample_nodes:
        spl = nx.single_source_shortest_path_length(G_gcc, s)
        lengths.extend(spl.values())
    d_real = np.mean(lengths)

print(f'Real network: C = {C_real:.4f}, <d> = {d_real:.4f}')

# ER random graph baseline (100 runs)
n_runs = 100
C_rand_list, d_rand_list = [], []
for _ in tqdm(range(n_runs), desc='ER simulations'):
    G_rand = nx.erdos_renyi_graph(N, p_er, seed=None)
    gcc_r = max(nx.connected_components(G_rand), key=len)
    G_rand_cc = G_rand.subgraph(gcc_r).copy()
    if G_rand_cc.number_of_nodes() < 2:
        continue
    C_rand_list.append(nx.average_clustering(G_rand_cc))
    # Fast approximation
    s_nodes = np.random.choice(list(G_rand_cc.nodes), size=min(20, G_rand_cc.number_of_nodes()), replace=False)
    l_list = []
    for s in s_nodes:
        spl = nx.single_source_shortest_path_length(G_rand_cc, s)
        l_list.extend(spl.values())
    d_rand_list.append(np.mean(l_list) if l_list else np.nan)

C_rand = np.mean(C_rand_list)
d_rand = np.nanmean(d_rand_list)
sigma_sw = (C_real / C_rand) / (d_real / d_rand)

print(f'\nSmall-world test:')
print(f'  C_real = {C_real:.4f}   | C_random = {C_rand:.4f}   | ratio = {C_real/C_rand:.2f}x')
print(f'  d_real = {d_real:.4f}   | d_random = {d_rand:.4f}   | ratio = {d_real/d_rand:.2f}x')
print(f'  σ (small-world coefficient) = {sigma_sw:.3f}')
if sigma_sw > 1:
    print('  → SMALL-WORLD network confirmed (σ > 1)')

sw_df = pd.DataFrame({
    'metric': ['Clustering C', 'Avg path length <d>', 'Small-world σ'],
    'real':   [C_real, d_real, sigma_sw],
    'ER_mean':[C_rand, d_rand, 1.0]
})
sw_df.to_csv(os.path.join(TABLES, 'p3_small_world.csv'), index=False)
print(sw_df.to_string(index=False))

## Step 3.3 – Centrality Analysis

In [ ]:
print('Computing centrality measures...')
deg_centrality  = nx.degree_centrality(G_full)
btw_centrality  = nx.betweenness_centrality(G_full, weight='weight', normalized=True)
eig_centrality  = nx.eigenvector_centrality(G_full, weight='weight', max_iter=1000)

centrality_df = pd.DataFrame({
    'country': list(deg_centrality.keys()),
    'degree_centrality': [deg_centrality[c] for c in deg_centrality],
    'betweenness_centrality': [btw_centrality[c] for c in deg_centrality],
    'eigenvector_centrality': [eig_centrality[c] for c in deg_centrality]
})

centrality_df['degree_rank']      = centrality_df['degree_centrality'].rank(ascending=False).astype(int)
centrality_df['betweenness_rank'] = centrality_df['betweenness_centrality'].rank(ascending=False).astype(int)
centrality_df['eigenvector_rank'] = centrality_df['eigenvector_centrality'].rank(ascending=False).astype(int)

centrality_df.sort_values('betweenness_rank').to_csv(
    os.path.join(TABLES, 'p3_centrality.csv'), index=False)

# India's ranks
india_row = centrality_df[centrality_df['country'] == 'India']
print('\nIndia centrality ranks:')
if not india_row.empty:
    print(india_row[['country','degree_rank','betweenness_rank','eigenvector_rank']].to_string(index=False))
else:
    print('  India not in network — checking alternative names...')
    india_like = [c for c in centrality_df['country'] if 'india' in c.lower() or 'India' in c]
    print('  Found:', india_like)

print('\nTop 15 by betweenness centrality:')
print(centrality_df.nsmallest(15, 'betweenness_rank')[
    ['country','degree_rank','betweenness_rank','eigenvector_rank']].to_string(index=False))

In [ ]:
# Centrality comparison radar/bar chart
highlight = ['United States of America','China','Russia','India',
             'Brazil','South Africa','Germany','Nigeria']
highlight = [c for c in highlight if c in centrality_df['country'].values]

h_df = centrality_df[centrality_df['country'].isin(highlight)].copy()
# Shorten names
name_map = {'United States of America': 'USA', 'South Africa': 'S.Africa'}
h_df['short_name'] = h_df['country'].map(lambda x: name_map.get(x, x))
h_df = h_df.sort_values('betweenness_centrality', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Centrality Comparison — Major Powers', color='white', fontsize=13)

metrics = ['degree_centrality', 'betweenness_centrality', 'eigenvector_centrality']
metric_labels = ['Degree', 'Betweenness', 'Eigenvector']
colors = ['#58a6ff', '#f85149', '#3fb950']

for ax, metric, label, color in zip(axes, metrics, metric_labels, colors):
    bars = ax.barh(h_df['short_name'], h_df[metric], color=color, alpha=0.85)
    ax.set_xlabel(f'{label} Centrality')
    ax.set_title(f'{label} Centrality', color='white')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'p3_centrality.png'), bbox_inches='tight', facecolor='#0d1117')
plt.show()

## Step 3.4 – Hierarchical Structure Test: C(k) vs k

In [ ]:
from scipy import stats as scipy_stats

def ck_analysis(G, label=''):
    """Compute C(k) — average clustering as a function of degree."""
    c_dict = nx.clustering(G)
    rows = []
    for n in G.nodes:
        rows.append({'k': G.degree(n), 'c': c_dict[n]})
    deg_cluster_df = pd.DataFrame(rows)
    # Bin by degree
    binned = deg_cluster_df.groupby('k')['c'].mean().reset_index()
    binned = binned[binned['k'] >= 2]   # skip k=0,1
    return binned

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Hierarchical Structure Test: C(k) vs k', color='white', fontsize=13)

eras_to_plot = [('full', '#58a6ff'), ('cold_war', '#f85149'),
                ('post_cw', '#3fb950'), ('post_9_11', '#d2a8ff'), ('recent', '#ffa657')]

for (era, color) in eras_to_plot:
    G_era = load_graph(era)
    ck = ck_analysis(G_era)
    if len(ck) < 3:
        continue
    log_k = np.log10(ck['k'].clip(lower=1))
    log_c = np.log10(ck['c'].clip(lower=1e-6))
    valid = np.isfinite(log_k) & np.isfinite(log_c)
    if valid.sum() >= 3:
        slope, intercept, r, p, _ = scipy_stats.linregress(log_k[valid], log_c[valid])
    else:
        slope = np.nan
    label_str = f'{era} (β={slope:.2f})' if not np.isnan(slope) else era
    axes[0].scatter(ck['k'], ck['c'], s=20, alpha=0.6, color=color, label=label_str)
    axes[1].scatter(log_k, log_c, s=20, alpha=0.6, color=color, label=label_str)
    if not np.isnan(slope):
        x_fit = np.linspace(log_k[valid].min(), log_k[valid].max(), 50)
        axes[1].plot(x_fit, intercept + slope*x_fit, '--', color=color, lw=1.5, alpha=0.8)

axes[0].set_xlabel('Degree k'); axes[0].set_ylabel('Mean clustering C(k)')
axes[0].set_title('C(k) vs k (linear)', color='white'); axes[0].legend(fontsize=7)

axes[1].set_xlabel('log₁₀(k)'); axes[1].set_ylabel('log₁₀ C(k)')
axes[1].set_title('C(k) vs k (log-log) — slope ≈ −1 → hierarchical', color='white')
axes[1].legend(fontsize=7)
# Reference line slope = -1
x_ref = np.linspace(0.5, 2.0, 50)
axes[1].plot(x_ref, -x_ref + 0.5, 'w--', lw=1, alpha=0.4, label='slope=-1 ref')

plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'p3_hierarchical_ck.png'), bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved C(k) hierarchical plot')

In [ ]:
# Compile full topology summary table
topo_rows = []
for era in ['full','cold_war','post_cw','post_9_11','recent']:
    G = load_graph(era)
    gcc_n = max(nx.connected_components(G), key=len)
    G_gcc = G.subgraph(gcc_n).copy()
    degs = [d for _,d in G.degree()]
    topo_rows.append({
        'era': era,
        'N': G.number_of_nodes(),
        'M': G.number_of_edges(),
        'density': round(nx.density(G), 4),
        'avg_degree': round(np.mean(degs), 2),
        'max_degree': max(degs),
        'clustering_C': round(nx.average_clustering(G), 4),
        'n_components': nx.number_connected_components(G)
    })

topo_df = pd.DataFrame(topo_rows)
topo_df.to_csv(os.path.join(TABLES, 'p3_topology.csv'), index=False)
print(topo_df.to_string(index=False))

## ✅ Phase 3 Complete
**Key Findings:**
- Power-law exponent γ (check if 2 < γ < 3 → scale-free)
- Small-world coefficient σ (check if σ > 1)
- C(k) slope (check if ≈ −1 → hierarchical)
- India rank on betweenness, degree, eigenvector

**→ Proceed to Notebook 04: Community Detection**